# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

 **My Rule:**<br> If CTR is low, engagement is low, low scroll event rate, position is greater than or equal to 10 and page is not updated for a quite long time than refresh page.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("""
Reason codes:
- low_ctr_and_stale — CTR is low, position is 10+, and content hasn't been updated in a long time
- low_engagement_and_low_scroll — engagement rate and scroll rate are both low, position is 10+
- all_signals_weak — CTR, engagement, and scroll are all low, position is 10+, and content is stale (the full match on your rule)
- stable — none of the above conditions are met, no refresh needed
""")


Reason codes:
- low_ctr_and_stale — CTR is low, position is 10+, and content hasn't been updated in a long time
- low_engagement_and_low_scroll — engagement rate and scroll rate are both low, position is 10+
- all_signals_weak — CTR, engagement, and scroll are all low, position is 10+, and content is stale (the full match on your rule)
- stable — none of the above conditions are met, no refresh needed



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

<h2>2.1: Import Libraries</h2>

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import os, getpass
import duckdb

<h2>2.2: Get data from hugging face Flyrank repo</h2>

In [3]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


<h2>2.3: Taken sample 90day window data from fact_daily</h2>

In [4]:
clients_last_3m = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position,
    ga4_pageviews, ga4_sessions, ga4_engaged_sessions, scroll_events
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01'
      AND report_date <= '2026-05-30'
      AND client_has_gsc == 'true'
      AND client_has_ga4 == 'true'
      AND gsc_data_available == 'true'
      AND ga4_data_available == 'true'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

<h2></h2>

<h2>Checking length</h2>

In [5]:
print(len(clients_last_3m))

1475963


<h2>Check columns</h2>

In [6]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events'],
      dtype='object')

<h2>Check NULL impressions if any</h2>

In [7]:
clients_last_3m["gsc_impressions"].isna().sum()

np.int64(0)

<h2>2.4: Load dim_content table data in pandas dataframe</h2>

In [8]:
dimf_content = con.sql(f"""
    SELECT client_hash_id, content_hash_id, content_updated_date
    FROM {TABLES['dim_content']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

<h2>2.5: Calculate CTR, Engagement rate, Scroll rate, Days since update</h2>

In [9]:
reference_date = dimf_content["content_updated_date"].max()
clients_last_3m["ctr"] = clients_last_3m["gsc_clicks"] / clients_last_3m["gsc_impressions"]
clients_last_3m["engagement_rate"] = clients_last_3m["ga4_engaged_sessions"] / clients_last_3m["ga4_sessions"]
clients_last_3m["scroll_rate"] = clients_last_3m["scroll_events"] / clients_last_3m["ga4_pageviews"].replace(0, pd.NA)
clients_last_3m = clients_last_3m.merge(
    dimf_content,
    on=("client_hash_id", "content_hash_id"),
    how="left"
)
clients_last_3m["days_since_update"] = (reference_date - clients_last_3m["content_updated_date"]).dt.days

<h2>Checking columns</h2>

In [10]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events', 'ctr', 'engagement_rate',
       'scroll_rate', 'content_updated_date', 'days_since_update'],
      dtype='object')

<h2>2.6: Drop unnecessary columns</h2>

In [11]:
clients_last_3m = clients_last_3m.drop(columns=['content_updated_date', 'gsc_clicks', 'ga4_pageviews', 'ga4_sessions',
       'ga4_engaged_sessions', 'scroll_events'])

<h2>Checking columns</h2>

In [12]:
clients_last_3m.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
       'days_since_update'],
      dtype='object')

<h2>2.7:
2 Signals: <br>  
1. CTR-vs-position (FlyRank signal)<br>
2. Engagement rate vs Scroll rate </h2>

In [13]:
# SIGNAL 1:
clients_last_3m["position_bucket"] = pd.cut(
    clients_last_3m["gsc_avg_position"],
    bins=[0, 3, 10, 20, 100, 100000],
    labels=["1-3", "4-10", "11-20", "21-100", "100+"]
)

signal1 = clients_last_3m.groupby("position_bucket").agg(
    avg_ctr=("ctr", "mean"),
    n=("content_hash_id", "count")
)
print("=== Signal 1: CTR-vs-position ===")
print(signal1)
print("Verdict: MIXED if avg_ctr decreases as position bucket gets worse but we can see if position crosses 100 then there is increase in average ctr")

# SIGNAL 2:
clients_last_3m["engagement_bucket"] = pd.cut(
    clients_last_3m["engagement_rate"],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=["low", "medium", "high", "very_high"]
)

signal2 = clients_last_3m.groupby("engagement_bucket").agg(
    avg_scroll_rate=("scroll_rate", "mean"),
    n=("content_hash_id", "count")
)
print("=== Signal 2: Engagement rate vs Scroll rate ===")
print(signal2)
print("Verdict: CONFIRMED if engagement is high the average scroll rate will also be high")

/tmp/ipykernel_3063/1362672175.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1 = clients_last_3m.groupby("position_bucket").agg(


=== Signal 1: CTR-vs-position ===
                  avg_ctr       n
position_bucket                  
1-3              0.026993  109849
4-10             0.013689  672228
11-20            0.011248  309390
21-100           0.006666  374920
100+             0.036157     380
Verdict: MIXED if avg_ctr decreases as position bucket gets worse but we can see if position crosses 100 then there is increase in average ctr


/tmp/ipykernel_3063/1362672175.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2 = clients_last_3m.groupby("engagement_bucket").agg(


=== Signal 2: Engagement rate vs Scroll rate ===
                  avg_scroll_rate      n
engagement_bucket                       
low                      0.167833  33909
medium                    0.37699  31483
high                     0.537507   1014
very_high                0.807498  35525
Verdict: CONFIRMED if engagement is high the average scroll rate will also be high


<h2>2.8: Check details of columns = ctr, engagement_rate, scroll_rate, days_since_update to find threshold values</h2>

In [14]:
print(clients_last_3m[clients_last_3m["ctr"] < 0.000005])

        report_date           client_hash_id           content_hash_id  \
0        2026-03-01  client_65de48885f4ef01b  content_5c80451459c29b4a   
1        2026-03-01  client_65de48885f4ef01b  content_b1f61fc81b28b2d4   
2        2026-03-01  client_65de48885f4ef01b  content_e25ea7297a1dffd3   
3        2026-03-01  client_65de48885f4ef01b  content_6b0149a80607dac3   
5        2026-03-01  client_65de48885f4ef01b  content_872342e050545a12   
...             ...                      ...                       ...   
1475953  2026-05-30  client_1a8bf67cad4ee525  content_3811343b165eb63a   
1475956  2026-05-30  client_1a8bf67cad4ee525  content_fadf7ae978fd082e   
1475959  2026-05-30  client_1a8bf67cad4ee525  content_5eb9c0b1de0202d2   
1475960  2026-05-30  client_1a8bf67cad4ee525  content_349c92d5cd468777   
1475961  2026-05-30  client_1a8bf67cad4ee525  content_978d1979c92d5bdb   

         gsc_impressions  gsc_avg_position  ctr  engagement_rate scroll_rate  \
0                      5       

In [15]:
clients_last_3m["engagement_rate"].value_counts()

,count
engagement_rate,
0.000000,1353496
1.000000,35508
0.500000,19020
0.333333,11135
0.250000,7469
...,...
0.003210,1
0.125786,1
0.029126,1


In [16]:
clients_last_3m["scroll_rate"].value_counts()

,count
scroll_rate,
0.0,1183288
1.0,87224
0.5,63002
0.333333,23945
0.25,22008
...,...
0.008754,1
0.028302,1
0.00786,1


In [17]:
clients_last_3m["days_since_update"].value_counts()

,count
days_since_update,
47,335993
131,237582
25,109276
49,89172
19,80452
...,...
362,3
74,3
255,2


<h2>2.9: Threshold values</h2>

In [18]:
ctr_threshold = 0.00005
engagement_threshold = 0
scroll_threshold = 0
staleness_threshold = 30

<h2>2.10: Calculate decline_score proxy label</h2>

In [19]:
clients_last_3m["decline_score"] = (
    (clients_last_3m["ctr"] < ctr_threshold).astype(int) +
    (clients_last_3m["engagement_rate"] <= engagement_threshold).astype(int) +
    (clients_last_3m["scroll_rate"] <= scroll_threshold).astype(int) +
    (clients_last_3m["gsc_avg_position"] >= 10).astype(int) +
    (clients_last_3m["days_since_update"] > staleness_threshold).astype(int)
)

<h2>2.11: Sort decline_score proxy label in descending order for data window</h2>

In [20]:
ranked_queue = clients_last_3m.sort_values("decline_score", ascending=False)

<h2>2.12: Print top 10 ranked rows</h2>

In [21]:
print("Top 10")
ranked_queue[:10]

Top 10


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
1475959,2026-05-30,client_1a8bf67cad4ee525,content_5eb9c0b1de0202d2,9,17.444444,0.0,0.0,0.0,35,11-20,NaN,5
8,2026-03-01,client_65de48885f4ef01b,content_4c185d1c173cd53d,23,30.304348,0.0,0.0,0.0,131,21-100,NaN,5
1475942,2026-05-30,client_1a8bf67cad4ee525,content_5eb560fef36a11e6,1,11.000000,0.0,0.0,0.0,35,11-20,NaN,5
1475945,2026-05-30,client_1a8bf67cad4ee525,content_2c488c733b8c2220,15,11.666667,0.0,0.0,0.0,35,11-20,NaN,5
19,2026-03-01,client_c182d11e4862a37d,content_da76e1818babb4ce,132,11.348485,0.0,0.0,0.0,47,11-20,NaN,5
18,2026-03-01,client_c182d11e4862a37d,content_f4a0e5c90b283626,250,19.448000,0.0,0.0,0.0,47,11-20,NaN,5
17,2026-03-01,client_c182d11e4862a37d,content_d926564dfe83536b,47,31.595745,0.0,0.0,0.0,47,21-100,NaN,5
598710,2026-04-16,client_23a62021009f63c4,content_02f32646ff66bb41,5,20.400000,0.0,0.0,0.0,47,21-100,NaN,5
1475914,2026-05-30,client_1a8bf67cad4ee525,content_92af9b4ca5a68926,16,11.687500,0.0,0.0,0.0,47,11-20,NaN,5
598726,2026-04-16,client_23a62021009f63c4,content_e3df741ab59b92ab,84,29.476190,0.0,0.0,0.0,47,21-100,NaN,5


<h2>2.13: Save ranked queue data in 'CSV' format in 'work/outputs' directory and locak machine</h2>

In [22]:
from google.colab import files

# Ensure the output directory exists
os.makedirs('work/outputs', exist_ok=True)

# Save the ranked queue to CSV
output_path = 'work/outputs/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)

print(f'File saved to {output_path}. Starting download...')

# Trigger browser download to local machine
files.download(output_path)

File saved to work/outputs/baseline_action_score.csv. Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

<h2>3.1: Top 20</h2>

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ranked_queue[:20]

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
1475959,2026-05-30,client_1a8bf67cad4ee525,content_5eb9c0b1de0202d2,9,17.444444,0.0,0.0,0.0,35,11-20,NaN,5
8,2026-03-01,client_65de48885f4ef01b,content_4c185d1c173cd53d,23,30.304348,0.0,0.0,0.0,131,21-100,NaN,5
1475942,2026-05-30,client_1a8bf67cad4ee525,content_5eb560fef36a11e6,1,11.000000,0.0,0.0,0.0,35,11-20,NaN,5
1475945,2026-05-30,client_1a8bf67cad4ee525,content_2c488c733b8c2220,15,11.666667,0.0,0.0,0.0,35,11-20,NaN,5
19,2026-03-01,client_c182d11e4862a37d,content_da76e1818babb4ce,132,11.348485,0.0,0.0,0.0,47,11-20,NaN,5
18,2026-03-01,client_c182d11e4862a37d,content_f4a0e5c90b283626,250,19.448000,0.0,0.0,0.0,47,11-20,NaN,5
17,2026-03-01,client_c182d11e4862a37d,content_d926564dfe83536b,47,31.595745,0.0,0.0,0.0,47,21-100,NaN,5
598710,2026-04-16,client_23a62021009f63c4,content_02f32646ff66bb41,5,20.400000,0.0,0.0,0.0,47,21-100,NaN,5
1475914,2026-05-30,client_1a8bf67cad4ee525,content_92af9b4ca5a68926,16,11.687500,0.0,0.0,0.0,47,11-20,NaN,5
598726,2026-04-16,client_23a62021009f63c4,content_e3df741ab59b92ab,84,29.476190,0.0,0.0,0.0,47,21-100,NaN,5


<h2>My Observation</h2>

<h2>3.2: Action: "Refresh_needed" because decline score is highest </h2>

<h2>3.3: Reason code: <br>
All signals weak for all 20 rows</h2>

<h2>3.4: Confidence note:</h2>
1. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 21-100 bucket. The day_since_update is also > 30. And impression is just 1.<br>
2. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in > 10 bucket. The day_since_update is also > 30. But impressions are somewhat there<br>
3. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in > 10 bucket. The day_since_update is also > 30. But impressions are somewhat there.<br>
4. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 21-100 bucket. The day_since_update is also > 100. <br>
5. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in > 10 bucket. The day_since_update is also > 30. But impressions are 54 which are somewhat reasonable for this confidence.<br>
6. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies around 10 bucket. The day_since_update is also > 30. Because of where it lies in position medium confidence.<br>
7.  Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in > 10 bucket. The day_since_update is also > 30. The confidence is this because impressions are somewhat reasonable. <br>
8. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 21-100 bucket. The day_since_update is also > 30. Impressions are also very less. <br>
9. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 21-100 bucket. The day_since_update is also > 30. Impressions are also very less.<br>
10. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 21-100 bucket. The day_since_update is also > 30. Impressions are 67 and so the above 0 values are somewhat problamatic.<br>
11. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and impressions are low as well as position.<br>
12. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and avg. position is fine but low impressions.<br>
13. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and impressions are low as well as position.<br>
14. Medium confidence: As ctr, engagement rate and scroll_rate are 0 each, and avg. position is lower but somewhat reasonable impressions.<br>
15. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and impressions are low and position is lower also.<br>
16. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and impressions are low but position is close to 10 but due to other strong factors high confidence.<br>
17. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 11-20 bucket. The day_since_update is also > 100.<br>
18. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 11-20 bucket. The day_since_update is also > 100.<br>
19. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 11-20 bucket. The day_since_update is also > 100.<br>
20. High confidence: As ctr, engagement rate and scroll_rate are 0 each, and position also lies in 11-20 bucket. The day_since_update is also > 100.


<h2>3.5: What would make it wrong?</h2>
1. Would be wrong if this page simply hasn't accumulated enough traffic yet to judge — not a real performance problem.<br>
2. Would be wrong if the page is on the edge of page 1 and just needs time, not a content refresh.<br>
3. Would be wrong if 38 days is too soon to call this "long neglected" — may just be normal maturation lag.<br>
4. Would be wrong if this page's low traffic is due to a niche/low-demand topic rather than declining quality.<br>
5. Would be wrong if this traffic came from a single anomalous spike (e.g., bot traffic or a one-off referral) rather than sustained real visits.<br>
6. Would be wrong if position is measurement noise right at the 10/11 boundary, making the "position ≥ 10" trigger arbitrary here.<br>
7. Would be wrong if 28 impressions with genuinely 0 engagement reflects a mismatched search intent, not something a refresh would fix.<br>
8. Would be wrong if this page is simply new/low-priority and not worth refresh effort yet.<br>
9. Would be wrong if low volume reflects a legitimately low-demand keyword, not decaying content.<br>
10. Would be wrong if impressions are inflated by irrelevant/broad-match queries that were never going to convert regardless of content quality.<br>
11. Would be wrong if this is simply a poor keyword-content match rather than a staleness issue.<br>
12. Would be wrong if the page is too new for its traffic to have stabilized.<br>
13. Would be wrong if low volume reflects niche topic demand, not quality decay.<br>
14. Would be wrong if this page's topic naturally has zero-click search behavior (e.g., answers fully shown in the snippet).<br>
15. Would be wrong if this page gets almost no search demand regardless of freshness.<br>
16. Would be wrong if the ranking position itself is unstable/fluctuating day to day, making this single snapshot misleading.<br>
17. Would be wrong if low impressions reflect seasonal/cyclical demand rather than a real decline.<br>
18. Would be wrong if position (10.4) is borderline enough that this page is essentially performing fine.<br>
19. Would be wrong if this reflects low search demand for the topic, not neglect.<br>
20. Would be wrong if position (10.4) is right at the boundary and effectively equivalent to a "good" ranking.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

<p> Weak picks: 1, 8, 12, 13, 15, 19</p>
<p> Reason: Because impressions are there very few and statistically it would be difficult to predict whether refresh needed or not as ctr, engagement rate, scroll rate will be less as impressions are less or maybe 0, as in this case.</p>

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak_picks = ranked_queue[:20].copy()

In [25]:
weak_picks[weak_picks['gsc_impressions'] < 10]

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
1475959,2026-05-30,client_1a8bf67cad4ee525,content_5eb9c0b1de0202d2,9,17.444444,0.0,0.0,0.0,35,11-20,NaN,5
1475942,2026-05-30,client_1a8bf67cad4ee525,content_5eb560fef36a11e6,1,11.000000,0.0,0.0,0.0,35,11-20,NaN,5
598710,2026-04-16,client_23a62021009f63c4,content_02f32646ff66bb41,5,20.400000,0.0,0.0,0.0,47,21-100,NaN,5
598725,2026-04-16,client_23a62021009f63c4,content_27aefb596af51514,7,23.142857,0.0,0.0,0.0,47,21-100,NaN,5
598705,2026-04-16,client_23a62021009f63c4,content_5f61d45d72e0b152,5,29.800000,0.0,0.0,0.0,47,21-100,NaN,5
598704,2026-04-16,client_23a62021009f63c4,content_5f5f80f37c39c312,1,19.000000,0.0,0.0,0.0,47,11-20,NaN,5
598700,2026-04-16,client_23a62021009f63c4,content_648be9d432a4eba4,3,65.000000,0.0,0.0,0.0,47,21-100,NaN,5
598697,2026-04-16,client_23a62021009f63c4,content_9dba3c0f79985b7c,5,15.600000,0.0,0.0,0.0,47,11-20,NaN,5
598695,2026-04-16,client_23a62021009f63c4,content_ef6f78ab86fc9b46,7,61.714286,0.0,0.0,0.0,47,21-100,NaN,5


In [26]:
no_flags_columns = ranked_queue.copy()

In [27]:
no_flags_columns.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
       'days_since_update', 'position_bucket', 'engagement_bucket',
       'decline_score'],
      dtype='object')

In [28]:
no_leakage_window= ranked_queue.copy()

In [29]:
no_leakage_window["report_date"].max()

Timestamp('2026-05-30 00:00:00')

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.